# Live hybrid-cache correctness end to end

This notebook is a runnable wrapper around `benchmarks.hybrid_cache_correctness.live_e2e`, so CLI and notebook runs use the same capture and comparison implementation. Start LMCache and vLLM with `LMCacheMPConnector`, `--no-enable-prefix-caching`, and `--return-tokens-as-token-ids` before running all cells. The final cell captures two deterministic exact-prefix runs, writes the traces outside the source tree, and fails if any output, request, KV-content, or lifecycle evidence diverges.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Standard
from pathlib import Path
from urllib.request import urlopen
import json
import os
import shlex
import subprocess
import sys

served_model = os.environ.get("VLLM_SERVED_MODEL_NAME", "Qwen/Qwen3-0.6B")
hf_model = os.environ.get("HF_MODEL_NAME", served_model)
cache_model = os.environ.get("LMCACHE_MODEL_NAME", served_model)
vllm_url = os.environ.get("VLLM_URL", "http://localhost:8000")
lmcache_url = os.environ.get("LMCACHE_URL", "http://localhost:8080")
default_output = "/tmp/lmcache-hybrid-correctness-e2e"
output_dir = Path(os.environ.get("LMCACHE_E2E_OUTPUT_DIR", default_output))

In [ ]:
for health_url in (f"{lmcache_url}/healthcheck", f"{vllm_url}/v1/models"):
    with urlopen(health_url, timeout=10) as response:  # noqa: S310
        assert response.status == 200, (health_url, response.status)
print("LMCache and vLLM health checks passed")

In [ ]:
repo_candidates = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in repo_candidates if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("run from the LMCache repository")
command = [
    sys.executable,
    "-m",
    "benchmarks.hybrid_cache_correctness.live_e2e",
    "--model",
    served_model,
    "--hf-model",
    hf_model,
    "--cache-model",
    cache_model,
    "--vllm-url",
    vllm_url,
    "--lmcache-url",
    lmcache_url,
    "--prompt-tokens",
    "1024",
    "--steps",
    "4",
    "--top-k",
    "5",
    "--output-dir",
    str(output_dir),
]
print("$", shlex.join(command))
completed = subprocess.run(
    command, check=True, capture_output=True, text=True, cwd=repo_root
)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
print(completed.stdout)
start = completed.stdout.find("{")
if start < 0:
    raise RuntimeError("live driver did not emit JSON evidence")
evidence = json.JSONDecoder().raw_decode(completed.stdout[start:])[0]
assert evidence["matched"] is True
assert evidence["first_divergence"] is None
assert evidence["frames"] == 4
for artifact in ("reference.json", "candidate.json", "report.json"):
    assert (output_dir / artifact).is_file(), artifact
print(f"Exact-prefix traces matched; artifacts: {output_dir}")